# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.0 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID='task324'
TASK_JSON=Path(COMPETITION)/'task324.json'
OUT_DIR=Path.cwd()/'task324_rare_seed_diagonal_onnx'
OUT_DIR.mkdir(exist_ok=True)
ONNX_PATH=OUT_DIR/f'{TASK_ID}.onnx'
SUMMARY_PATH=OUT_DIR/f'{TASK_ID}_validation_summary.json'
SUBMISSION_PATH=Path.cwd()/'submission.zip'
N_COLORS=10; MAX_H=30; MAX_W=30
RARE_MAX=4.5


In [6]:
def grid_to_tensor(grid, full_background=False):
    arr=np.array(grid,dtype=np.int64)
    t=np.zeros((1,N_COLORS,MAX_H,MAX_W),dtype=np.float32)
    if full_background:
        t[:,0,:,:]=1.0
    h,w=arr.shape
    for r in range(h):
        for c in range(w):
            t[0,:,r,c]=0.0
            t[0,int(arr[r,c]),r,c]=1.0
    return t

class Task324RareSeedDiagonal(nn.Module):
    def __init__(self):
        super().__init__()
        rr=torch.arange(MAX_H,dtype=torch.float32).view(MAX_H,1).expand(MAX_H,MAX_W)
        cc=torch.arange(MAX_W,dtype=torch.float32).view(1,MAX_W).expand(MAX_H,MAX_W)
        self.register_buffer('diff',rr-cc)
        self.register_buffer('summ',rr+cc)
        self.register_buffer('colors',torch.arange(N_COLORS,dtype=torch.float32).view(N_COLORS,1,1))
        self.register_buffer('zeros_row', torch.zeros(1,MAX_W,dtype=torch.float32))
        self.register_buffer('zeros_col', torch.zeros(MAX_H,1,dtype=torch.float32))

    def _neighbor4(self, img):
        up=torch.cat([self.zeros_row.to(img.device), img[:-1,:]], dim=0)
        down=torch.cat([img[1:,:], self.zeros_row.to(img.device)], dim=0)
        left=torch.cat([self.zeros_col.to(img.device), img[:,:-1]], dim=1)
        right=torch.cat([img[:,1:], self.zeros_col.to(img.device)], dim=1)
        return up+down+left+right

    def forward(self,x):
        x0=x[0]
        active=x0.sum(dim=0)>0.5
        v=(x0*self.colors).sum(dim=0)
        counts=x0.sum(dim=(1,2))
        rare=(counts>0.5) & (counts<=RARE_MAX)
        rare_mask=(x0*rare.to(torch.float32).view(N_COLORS,1,1)).sum(dim=0)>0.5
        marker=active & rare_mask

        line=torch.zeros((MAX_H,MAX_W),dtype=torch.bool,device=x.device)
        for d in range(-(MAX_H-1),MAX_W):
            dval=torch.tensor(float(d),dtype=torch.float32,device=x.device)
            dmask=torch.abs(self.diff-dval)<0.25
            has_seed=(marker & dmask).float().sum()>0.5
            line=line|(has_seed & dmask)
        for s in range(0,MAX_H+MAX_W-1):
            sval=torch.tensor(float(s),dtype=torch.float32,device=x.device)
            smask=torch.abs(self.summ-sval)<0.25
            has_seed=(marker & smask).float().sum()>0.5
            line=line|(has_seed & smask)

        outv=v
        # Each rare color learns the base color underneath it from the 4-neighbor majority.
        for m in range(N_COLORS):
            mval=torch.tensor(float(m),dtype=torch.float32,device=x.device)
            seed_m=(x0[m]>0.5) & rare[m]
            scores=[]
            for bg in range(N_COLORS):
                neigh=self._neighbor4(x0[bg])
                score=(seed_m.float()*neigh).sum()
                # Rare colors should not be selected as background carriers.
                score=torch.where(rare[bg], torch.tensor(-1.0,dtype=torch.float32,device=x.device), score)
                scores.append(score)
            scores=torch.stack(scores)
            best=torch.argmax(scores)
            best_score=scores.max()
            has_seed=(seed_m.float().sum()>0.5) & (best_score>0.5)
            for bg in range(N_COLORS):
                bg_is_best=(best==bg)
                outv=torch.where(line & active & (x0[bg]>0.5) & has_seed & bg_is_best, mval.expand(MAX_H,MAX_W), outv)

        channels=[]
        for k in range(N_COLORS):
            kval=torch.tensor(float(k),dtype=torch.float32,device=x.device)
            channels.append(((torch.abs(outv-kval)<0.25)&active).to(torch.float32))
        return torch.stack(channels,dim=0).unsqueeze(0)


def validate_torch(task,model):
    out={}
    for split in ['train','test','arc-gen']:
        ok=0; total=0; bad=[]
        for i,ex in enumerate(task.get(split,[])):
            if 'output' not in ex: continue
            h=len(ex['input']); w=len(ex['input'][0])
            if h>30 or w>30: continue
            x=torch.from_numpy(grid_to_tensor(ex['input']))
            with torch.no_grad():
                y=model(x).numpy()[0]
            pred=y.argmax(axis=0)[:h,:w]
            exp=np.array(ex['output'])
            good=np.array_equal(pred,exp)
            ok+=int(good); total+=1
            if not good and len(bad)<8:
                bad.append({'idx':i,'diff':int((pred!=exp).sum())})
        out[split]={'ok':ok,'total':total,'bad_first8':bad}
    return out

In [7]:
task=json.load(open(TASK_JSON))
model=Task324RareSeedDiagonal().eval()
print(validate_torch(task,model))
dummy=torch.from_numpy(grid_to_tensor(task['test'][0]['input']))
torch.onnx.export(model,dummy,str(ONNX_PATH),opset_version=17,input_names=['input'],output_names=['output'],dynamic_axes=None,dynamo=False)



{'train': {'ok': 3, 'total': 3, 'bad_first8': []}, 'test': {'ok': 1, 'total': 1, 'bad_first8': []}, 'arc-gen': {'ok': 262, 'total': 262, 'bad_first8': []}}


/tmp/ipykernel_16/89688901.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model,dummy,str(ONNX_PATH),opset_version=17,input_names=['input'],output_names=['output'],dynamic_axes=None,dynamo=False)
/tmp/ipykernel_16/3176428490.py:42: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  dval=torch.tensor(float(d),dtype=torch.float32,device=x.device)
/tmp/ipykernel_16/3176428490.py:47: Trac

In [8]:
onnx_model=onnx.load(str(ONNX_PATH)); onnx.checker.check_model(onnx_model)
ops=Counter(n.op_type for n in onnx_model.graph.node)
sess=ort.InferenceSession(str(ONNX_PATH),providers=['CPUExecutionProvider'])


In [9]:
def run_ort(split):
    ok=0; total=0; bad=[]; outside=0; activecov=0
    for i,ex in enumerate(task.get(split,[])):
        if 'output' not in ex: continue
        h=len(ex['input']); w=len(ex['input'][0])
        if h>30 or w>30: continue
        x=grid_to_tensor(ex['input'])
        p=sess.run(None,{'input':x})[0]
        pred=(p>0.5).astype(np.float32)
        exp=grid_to_tensor(ex['output'])
        good=np.array_equal(pred,exp)
        ok+=int(good); total+=1
        active=x.sum(axis=1,keepdims=True)>0.5
        outside+=int(np.all(pred*(~active)==0))
        activecov+=int(np.all(pred.sum(axis=1,keepdims=True)[active]==1.0))
        if not good and len(bad)<8:
            bad.append({'idx':i,'diff':int(np.abs(pred-exp).sum())})
    return {'ok':ok,'total':total,'bad_first8':bad,'outside_zero_ok':outside,'active_canvas_covered_ok':activecov}

summary={'task_id':TASK_ID,'model_family':'rare-seed diagonal field with local background-carrier mapping','input_shape':[1,10,30,30],'output_shape':[1,10,30,30],'onnx_size_bytes':ONNX_PATH.stat().st_size,'ops':dict(ops),'forbidden_ops':[op for op in ops if op in {'Loop','Scan','NonZero','Unique','Script','Function'}],'function_count':len(onnx_model.functions),'rare_threshold':RARE_MAX}

for split in ['train','test','arc-gen']:
    summary[split]=run_ort(split)
json.dump(summary,open(SUMMARY_PATH,'w'),indent=2)
print(json.dumps(summary,indent=2)[:1400])

assert summary['train']['ok']==summary['train']['total']
assert summary['test']['ok']==summary['test']['total']
assert summary['arc-gen']['ok']==summary['arc-gen']['total']
assert summary['onnx_size_bytes']<1_400_000
assert not summary['forbidden_ops']


{
  "task_id": "task324",
  "model_family": "rare-seed diagonal field with local background-carrier mapping",
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "onnx_size_bytes": 1111344,
  "ops": {
    "Identity": 398,
    "Constant": 885,
    "Gather": 21,
    "ReduceSum": 232,
    "Greater": 151,
    "Mul": 102,
    "LessOrEqual": 1,
    "And": 479,
    "Cast": 139,
    "Reshape": 11,
    "Abs": 128,
    "Less": 128,
    "Or": 118,
    "Slice": 40,
    "Concat": 411,
    "Add": 300,
    "Where": 200,
    "Unsqueeze": 111,
    "ArgMax": 10,
    "ReduceMax": 10,
    "Equal": 100,
    "Sub": 10
  },
  "forbidden_ops": [],
  "function_count": 0,
  "rare_threshold": 4.5,
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first8": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first8": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1
  },
  "a

In [10]:
with zipfile.ZipFile(SUBMISSION_PATH,'w',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH,arcname=f'{TASK_ID}.onnx')
print('wrote',SUBMISSION_PATH)

wrote /kaggle/working/submission.zip
